In [1]:
import numpy as np
from sklearn.datasets import fetch_20newsgroups
from sklearn.feature_extraction.text import CountVectorizer
from gensim import corpora, models
from gensim.utils import simple_preprocess
from gensim.models import CoherenceModel

# Из пдфки

In [2]:
# Определяем нужные группы
selected_classes = [
    'alt.atheism',
    'comp.graphics',
    'sci.space',
    'talk.politics.mideast'
]

# загружаем дс
newsgroups = fetch_20newsgroups(
        subset='all', 
        categories=selected_classes,
        remove=('headers','footers', 'quotes')
    )

In [3]:
# Preprocess the text data and create a list of tokenized documents
tokenized_documents = [simple_preprocess(text) for text in newsgroups.data]
# Create a dictionary mapping of words to unique IDs
dictionary = corpora.Dictionary(tokenized_documents)
# Create a Bag of Words (BoW) representation of the documents
bow_corpus = [dictionary.doc2bow(doc) for doc in tokenized_documents]
# Train the LDA model
lda_model = models.LdaModel(bow_corpus, num_topics=20, id2word=dictionary, passes=15)
# Print the topics and their top words
topics = lda_model.print_topics(num_words=10)
for topic in topics:
    print(topic)

(0, '0.018*"atheism" + 0.014*"alt" + 0.007*"rocketry" + 0.006*"ks" + 0.005*"pages" + 0.004*"rockets" + 0.004*"engines" + 0.004*"and" + 0.004*"faa" + 0.003*"author"')
(1, '0.005*"adams" + 0.005*"michael" + 0.004*"recommendations" + 0.004*"aipac" + 0.003*"nsmca" + 0.003*"alaska" + 0.003*"acad" + 0.003*"loser" + 0.002*"nick" + 0.002*"andi"')
(2, '0.046*"the" + 0.019*"to" + 0.013*"it" + 0.011*"of" + 0.011*"and" + 0.011*"is" + 0.011*"you" + 0.011*"that" + 0.008*"was" + 0.007*"this"')
(3, '0.027*"henrik" + 0.022*"bm" + 0.014*"armenia" + 0.012*"planes" + 0.011*"armenians" + 0.011*"azeris" + 0.010*"karabakh" + 0.009*"turkey" + 0.006*"stalin" + 0.006*"conflict"')
(4, '0.032*"adam" + 0.014*"harvard" + 0.012*"bob" + 0.012*"com" + 0.012*"tek" + 0.011*"vice" + 0.011*"das" + 0.010*"manhattan" + 0.010*"shostack" + 0.010*"beauchaine"')
(5, '0.078*"the" + 0.032*"of" + 0.027*"to" + 0.023*"and" + 0.023*"in" + 0.013*"that" + 0.012*"israel" + 0.011*"was" + 0.009*"for" + 0.009*"he"')
(6, '0.095*"the" + 0.05

In [4]:
document_topic_vectors = []
for doc_bow in bow_corpus:
    document_topics = lda_model.get_document_topics(doc_bow, minimum_probability=0.0)
    document_topic_vector = [topic_prob for _, topic_prob in document_topics]
    document_topic_vectors.append(document_topic_vector)


In [5]:
# Specify the number of documents to print
num_documents_to_print = 5
# Iterate through the documents and get their topic distributions
document_topic_vectors = []
for i, doc_bow in enumerate(bow_corpus):
    document_topics = lda_model.get_document_topics(doc_bow, minimum_probability=0.0)
    document_topic_vector = [topic_prob for _, topic_prob in document_topics]
    document_topic_vectors.append(document_topic_vector)
    # Print the topic vector for the first num_documents_to_print documents
    if i < num_documents_to_print:
        print(f"Document {i + 1} Topic Vector: {document_topic_vector}")

Document 1 Topic Vector: [np.float32(0.0038652734), np.float32(0.0038652734), np.float32(0.0038652734), np.float32(0.0038652734), np.float32(0.0038652734), np.float32(0.0038652734), np.float32(0.4642001), np.float32(0.0038652734), np.float32(0.0038652734), np.float32(0.0038652734), np.float32(0.0038652734), np.float32(0.324701), np.float32(0.0038652734), np.float32(0.0038652734), np.float32(0.0038652734), np.float32(0.0038652734), np.float32(0.0038652734), np.float32(0.0038652734), np.float32(0.0038652734), np.float32(0.1453891)]
Document 2 Topic Vector: [np.float32(4.8948223e-05), np.float32(4.8948223e-05), np.float32(4.8948223e-05), np.float32(4.8948223e-05), np.float32(4.8948223e-05), np.float32(4.8948223e-05), np.float32(0.11045901), np.float32(4.8948223e-05), np.float32(4.8948223e-05), np.float32(4.8948223e-05), np.float32(4.8948223e-05), np.float32(4.8948223e-05), np.float32(4.8948223e-05), np.float32(4.8948223e-05), np.float32(4.8948223e-05), np.float32(0.88865995), np.float32(4

## Смотрим параметры $\alpha$ и $\theta$

In [6]:
alpha_values = [0.01, 0.1, 0.5, 1.0, 2.0]
eta_values = [0.01, 0.1, 0.5, 1.0, 2.0]

num_topics = 20
results = []

In [7]:
# Полный перебор
for alpha in alpha_values:
    for eta in eta_values:
        print(f"Обучение: alpha={alpha}, eta={eta}")
        
        # Обучение модели
        lda_model = models.LdaModel(
            bow_corpus, 
            num_topics=num_topics, 
            id2word=dictionary, 
            alpha=alpha, 
            eta=eta,
            passes=15
        )
        
        # Оценка качества через coherence score
        coherence_model = CoherenceModel(
            model=lda_model, 
            texts=tokenized_documents, 
            dictionary=dictionary, 
            coherence='c_v'
        )
        coherence_score = coherence_model.get_coherence()
        
        results.append({
            'alpha': alpha,
            'eta': eta,
            'coherence': coherence_score
        })
        
        print(f"  Coherence Score: {coherence_score:.4f}\n")

Обучение: alpha=0.01, eta=0.01
  Coherence Score: 0.3992

Обучение: alpha=0.01, eta=0.1
  Coherence Score: 0.3928

Обучение: alpha=0.01, eta=0.5
  Coherence Score: 0.4634

Обучение: alpha=0.01, eta=1.0
  Coherence Score: 0.5106

Обучение: alpha=0.01, eta=2.0
  Coherence Score: 0.4246

Обучение: alpha=0.1, eta=0.01
  Coherence Score: 0.3596

Обучение: alpha=0.1, eta=0.1
  Coherence Score: 0.4246

Обучение: alpha=0.1, eta=0.5
  Coherence Score: 0.4623

Обучение: alpha=0.1, eta=1.0
  Coherence Score: 0.4733

Обучение: alpha=0.1, eta=2.0
  Coherence Score: 0.5155

Обучение: alpha=0.5, eta=0.01
  Coherence Score: 0.3510

Обучение: alpha=0.5, eta=0.1
  Coherence Score: 0.4186

Обучение: alpha=0.5, eta=0.5
  Coherence Score: 0.4218

Обучение: alpha=0.5, eta=1.0
  Coherence Score: 0.4550

Обучение: alpha=0.5, eta=2.0
  Coherence Score: 0.6937

Обучение: alpha=1.0, eta=0.01
  Coherence Score: 0.3449

Обучение: alpha=1.0, eta=0.1
  Coherence Score: 0.3700

Обучение: alpha=1.0, eta=0.5
  Coherenc

In [8]:
# Вывод результатов
print("\nРЕЗУЛЬТАТЫ:")
for r in sorted(results, key=lambda x: x['coherence'], reverse=True):
    print(f"alpha={r['alpha']}, eta={r['eta']} -> Coherence: {r['coherence']:.4f}")

# Лучшая комбинация
best = max(results, key=lambda x: x['coherence'])
print(f"\nЛучшие параметры: alpha={best['alpha']}, eta={best['eta']}")
print(f"Coherence Score: {best['coherence']:.4f}")


РЕЗУЛЬТАТЫ:
alpha=0.5, eta=2.0 -> Coherence: 0.6937
alpha=0.1, eta=2.0 -> Coherence: 0.5155
alpha=1.0, eta=0.5 -> Coherence: 0.5119
alpha=0.01, eta=1.0 -> Coherence: 0.5106
alpha=1.0, eta=2.0 -> Coherence: 0.5094
alpha=1.0, eta=1.0 -> Coherence: 0.4870
alpha=0.1, eta=1.0 -> Coherence: 0.4733
alpha=0.01, eta=0.5 -> Coherence: 0.4634
alpha=0.1, eta=0.5 -> Coherence: 0.4623
alpha=0.5, eta=1.0 -> Coherence: 0.4550
alpha=2.0, eta=0.5 -> Coherence: 0.4479
alpha=2.0, eta=1.0 -> Coherence: 0.4421
alpha=2.0, eta=2.0 -> Coherence: 0.4344
alpha=0.01, eta=2.0 -> Coherence: 0.4246
alpha=0.1, eta=0.1 -> Coherence: 0.4246
alpha=0.5, eta=0.5 -> Coherence: 0.4218
alpha=0.5, eta=0.1 -> Coherence: 0.4186
alpha=0.01, eta=0.01 -> Coherence: 0.3992
alpha=0.01, eta=0.1 -> Coherence: 0.3928
alpha=1.0, eta=0.1 -> Coherence: 0.3700
alpha=0.1, eta=0.01 -> Coherence: 0.3596
alpha=2.0, eta=0.01 -> Coherence: 0.3541
alpha=0.5, eta=0.01 -> Coherence: 0.3510
alpha=1.0, eta=0.01 -> Coherence: 0.3449
alpha=2.0, eta=0.

РЕЗУЛЬТАТЫ:

alpha=0.5, eta=2.0 -> Coherence: 0.6937

alpha=0.1, eta=2.0 -> Coherence: 0.5155

alpha=1.0, eta=0.5 -> Coherence: 0.5119

alpha=0.01, eta=1.0 -> Coherence: 0.5106

alpha=1.0, eta=2.0 -> Coherence: 0.5094

alpha=1.0, eta=1.0 -> Coherence: 0.4870

alpha=0.1, eta=1.0 -> Coherence: 0.4733

alpha=0.01, eta=0.5 -> Coherence: 0.4634

alpha=0.1, eta=0.5 -> Coherence: 0.4623

alpha=0.5, eta=1.0 -> Coherence: 0.4550

alpha=2.0, eta=0.5 -> Coherence: 0.4479

alpha=2.0, eta=1.0 -> Coherence: 0.4421

alpha=2.0, eta=2.0 -> Coherence: 0.4344

alpha=0.01, eta=2.0 -> Coherence: 0.4246

alpha=0.1, eta=0.1 -> Coherence: 0.4246

alpha=0.5, eta=0.5 -> Coherence: 0.4218

alpha=0.5, eta=0.1 -> Coherence: 0.4186

alpha=0.01, eta=0.01 -> Coherence: 0.3992

alpha=0.01, eta=0.1 -> Coherence: 0.3928

alpha=1.0, eta=0.1 -> Coherence: 0.3700

alpha=0.1, eta=0.01 -> Coherence: 0.3596

alpha=2.0, eta=0.01 -> Coherence: 0.3541

alpha=0.5, eta=0.01 -> Coherence: 0.3510

alpha=1.0, eta=0.01 -> Coherence: 0.3449

alpha=2.0, eta=0.1 -> Coherence: 0.3330



Лучшие параметры: alpha=0.5, eta=2.0

Coherence Score: 0.6937